In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib

In [9]:
df = pd.read_csv(r"E:\3-2\3200\\CrisisNet\Crisisnet\prediction\FloodPrediction.csv")
df = df.dropna(subset=['Flood?'])

print(df.head())
print(df.columns)

    Sl Station_Names  Year  Month  Max_Temp  Min_Temp  Rainfall  \
5    5       Barisal  1949      6      34.4      25.7     512.0   
6    6       Barisal  1949      7      33.4      25.8     575.0   
7    7       Barisal  1949      8      33.5      25.7     349.0   
16  16       Barisal  1950      5      36.1      25.5      87.0   
17  17       Barisal  1950      6      33.9      26.0     476.0   

    Relative_Humidity  Wind_Speed  Cloud_Coverage  Bright_Sunshine  \
5                80.0    1.631481             5.6         4.072340   
6                85.0    1.570370             5.4         3.738298   
7                86.0    1.329630             5.6         4.272340   
16               78.0    1.703704             4.8         7.046809   
17               87.0    1.631481             7.2         4.072340   

    Station_Number     X_COR     Y_COR  LATITUDE  LONGITUDE  ALT   Period  \
5            41950  536809.8  510151.9      22.7      90.36    4  1949.06   
6            41950  53

In [10]:
# 2. EDA & Safety Checks (Crucial for imbalanced data)
print("--- Missing Values Check ---")
print(df.isnull().sum())

print("\n--- Target Class Balance ---")
features = ['Rainfall', 'Relative_Humidity', 'Wind_Speed', 'Cloud_Coverage']
target = 'Flood?'
print(df[target].value_counts(normalize=True))

# 3. Safe Target Encoding
print(f"\nTarget unique values before mapping: {df[target].unique()}")
if df[target].dtype == 'object':
    df[target] = df[target].map({'Yes': 1, 'No': 0})

--- Missing Values Check ---
Sl                   0
Station_Names        0
Year                 0
Month                0
Max_Temp             0
Min_Temp             0
Rainfall             0
Relative_Humidity    0
Wind_Speed           0
Cloud_Coverage       0
Bright_Sunshine      0
Station_Number       0
X_COR                0
Y_COR                0
LATITUDE             0
LONGITUDE            0
ALT                  0
Period               0
Flood?               0
dtype: int64

--- Target Class Balance ---
Flood?
1.0    0.919653
0.0    0.080347
Name: proportion, dtype: float64

Target unique values before mapping: [1. 0.]


In [11]:
X = df[features]
y = df[target]

In [12]:
print(X)

       Rainfall  Relative_Humidity  Wind_Speed  Cloud_Coverage
5         512.0               80.0    1.631481             5.6
6         575.0               85.0    1.570370             5.4
7         349.0               86.0    1.329630             5.6
16         87.0               78.0    1.703704             4.8
17        476.0               87.0    1.631481             7.2
...         ...                ...         ...             ...
20528     496.0               87.0    2.000000             5.7
20536     753.0               85.0    3.700000             5.5
20537     821.0               87.0    3.500000             6.2
20538    1127.0               89.0    3.300000             6.5
20539     851.0               88.0    2.500000             6.2

[4493 rows x 4 columns]


In [13]:
print(y)

5        1.0
6        1.0
7        1.0
16       1.0
17       1.0
        ... 
20528    1.0
20536    1.0
20537    1.0
20538    1.0
20539    1.0
Name: Flood?, Length: 4493, dtype: float64


In [15]:
print(f"Rows: {len(df)}, y NaNs: {y.isnull().sum()}, y unique: {y.unique()}")

Rows: 4493, y NaNs: 0, y unique: [1. 0.]


In [16]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# 1. Split (stratified, since classes are imbalanced)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Pipeline with class-weight balancing
model_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(
        n_estimators=100, random_state=42, class_weight='balanced'
    ))
])

# 3. Train
model_pipeline.fit(X_train, y_train)

# 4. Evaluate on the held-out test set
predictions = model_pipeline.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, predictions) * 100:.2f}%")
print("\nClassification Report:\n", classification_report(y_test, predictions))
print("Confusion Matrix:\n", confusion_matrix(y_test, predictions))

# 5. Cross-validation for a more stable estimate (small minority class = single-split numbers are noisy)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model_pipeline, X, y, cv=cv, scoring='f1_macro')
print(f"\n5-Fold CV F1 (macro): {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

# 6. Feature importance — explains *why* the model decides what it decides
importances = model_pipeline.named_steps['classifier'].feature_importances_
for feat, imp in sorted(zip(features, importances), key=lambda x: -x[1]):
    print(f"{feat}: {imp:.3f}")

Accuracy: 98.44%

Classification Report:
               precision    recall  f1-score   support

         0.0       0.97      0.83      0.90        72
         1.0       0.99      1.00      0.99       827

    accuracy                           0.98       899
   macro avg       0.98      0.92      0.94       899
weighted avg       0.98      0.98      0.98       899

Confusion Matrix:
 [[ 60  12]
 [  2 825]]

5-Fold CV F1 (macro): 0.956 ± 0.012
Rainfall: 0.510
Relative_Humidity: 0.288
Cloud_Coverage: 0.178
Wind_Speed: 0.024


In [17]:
features_v2 = ['Rainfall', 'Relative_Humidity', 'Cloud_Coverage']
X2 = df[features_v2]
cv_scores_v2 = cross_val_score(model_pipeline, X2, y, cv=cv, scoring='f1_macro')
print(f"Without Wind_Speed — F1: {cv_scores_v2.mean():.3f} ± {cv_scores_v2.std():.3f}")

Without Wind_Speed — F1: 0.954 ± 0.010


In [18]:
features = ['Rainfall', 'Relative_Humidity', 'Cloud_Coverage']
target = 'Flood?'

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(
        n_estimators=100, random_state=42, class_weight='balanced'
    ))
])

model_pipeline.fit(X_train, y_train)

predictions = model_pipeline.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, predictions) * 100:.2f}%")
print("\nClassification Report:\n", classification_report(y_test, predictions))
print("Confusion Matrix:\n", confusion_matrix(y_test, predictions))

import joblib
filename = 'flood_prediction_pipeline.joblib'
joblib.dump(model_pipeline, filename)
print(f"Model saved as {filename}")

Accuracy: 98.33%

Classification Report:
               precision    recall  f1-score   support

         0.0       0.97      0.82      0.89        72
         1.0       0.98      1.00      0.99       827

    accuracy                           0.98       899
   macro avg       0.98      0.91      0.94       899
weighted avg       0.98      0.98      0.98       899

Confusion Matrix:
 [[ 59  13]
 [  2 825]]
Model saved as flood_prediction_pipeline.joblib
